# Исследование надежности заемщиков

Дополнение: интерактиный дашборд по проекту  доступен по <a href='https://public.tableau.com/app/profile/ruslan.karimov/viz/credit_dashboard_17300448507020/credit_dashboard' alt='Ссылка на дашборд'>ссылке</a>

<a class='anchor' id='content'></a>
## Содержание

1. [Описание проекта](#desc)
2. [Получение данных](#get_data)
3. [Первичный обзор данных](#review)
4. [Предобработка данных](#preproccesing)
5. [Ход исследования](#research)
6. [Результаты исследования](#result)
7. [Рекомендации по улучшению скоринга](#scoring)
8. [Рекомендации по сбору данных](#recomendatiton)

<a class='anchor' id='desc'></a>
### 1. Описание проекта

В нашем распоряжении данные одного из банков по платежеспособности клиентов.

#### Цель исследования
Выполнить предобработку данных с помощью 18 заданий и ответить на поставленные вопросы.

#### Задачи исследования
* есть ли зависимость между количеством детей и возвратом кредита в срок?
* есть ли зависимость между семейным положением и возвратом кредита в срок?
* есть ли зависимость между уровнем дохода и возвратом кредита в срок?
* как разные цели кредита влияют на его возврат в срок?
* приведите возможные причины появления пропусков в исходных данных.
* объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

#### Описание данных

* `children` - количество детей
* `days_employed` - стаж работы (дни)
* `dob_years` - возраст
* `education` - образование
* `education_id` - булевый тип образования: высшее или нет
* `family_status` - семейное положение
* `family_status_id` - ID семейного положения
* `gender` - пол
* `income_type` - тип занятости
* `debt` - признак ухода в просрочку
* `total_income` - совокупный доход
* `purpose` - цель кредита

#### Используемые библиотеки

* pandas
* seaborn
* matplotlib.pyplot
* os
* tableau
 
[_назад к содержанию_](#content)

<a class='anchor' id='get_data'></a>
### 2. Получение данных

Задание 1. Импортируйте необходимые модули и считайте данные из csv-файла в датафрейм, сохранив его в переменную `data`.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# домашняя директория проекта на локальной машине под управлением Windows
PROJECT_DIR = r"C:\Users\e6ton\PycharmProjects\yandex-practicum"

try:
    if os.path.exists(PROJECT_DIR):
        data = pd.read_csv(os.path.join(PROJECT_DIR, 'datasets', 'data.csv'))
    else:
        data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

[_назад к содержанию_](#content)

<a class='anchor' id='review'></a>
### 3. Первичный обзор данных

Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.

In [ ]:
data.head(20)

Задание 3. Выведите основную информацию о датафрейме.

In [ ]:
data.info()

[_назад к содержанию_](#content)

<a class='anchor' id='preproccesing'></a>
### 4. Предобработка данных

#### 4.1. Удаление пропусков

Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.

In [ ]:
data.isna().sum()

Задание 5. В двух столбцах есть пропущенные значения. Один из них — days_employed. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — total_income — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца income_type. Например, у человека с типом занятости сотрудник пропуск в столбце total_income должен быть заполнен медианным доходом среди всех записей с тем же типом.

In [ ]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

#### 4.2. Обработка аномальных значений

Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. Таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными.

In [ ]:
data['days_employed'] = data['days_employed'].abs()

Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.

In [ ]:
data.groupby('income_type')['days_employed'].agg('median')

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть.

Задание 8. Выведите перечень уникальных значений столбца `children`.

In [ ]:
data['children'].unique()

Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения.

In [ ]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.

In [ ]:
data['children'].unique()

#### 4.3. Удаление пропусков (продолжение)

Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.

In [ ]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.

In [ ]:
data.isna().sum()

#### 4.4. Изменение типов данных

Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный.

In [ ]:
data['total_income'] = data['total_income'].astype(int)

#### 4.5. Обработка дубликатов

Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру.

In [ ]:
data['education'] = data['education'].str.lower()

Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

#### 4.6. Категоризация данных

Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:

* 0–30000 — `'E'`;
* 30001–50000 — `'D'`;
* 50001–200000 — `'C'`;
* 200001–1000000 — `'B'`;
* 1000001 и выше — `'A'`.

Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()`.

In [ ]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        return 'unknown'


data['total_income_category'] = data['total_income'].apply(categorize_income)

Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.

In [ ]:
data['purpose'].unique()

Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:

* 'операции с автомобилем';
* 'операции с недвижимостью';
* 'проведение свадьбы';
* 'получение образования'.
  
Например, если в столбце `purpose` находится подстрока 'на покупку автомобиля', то в столбце `purpose_category` должна появиться строка 'операции с автомобилем'.

Используйте собственную функцию с именем `categorize_purpose()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.

In [ ]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'


data['purpose_category'] = data['purpose'].apply(categorize_purpose)

In [ ]:
data.groupby('purpose_category')['purpose_category'].count()

[_назад к содержанию_](#content)

<a class='anchor' id='research'></a>
### 5. Ход исследования

#### 5.1. Есть ли зависимость между количеством детей и возвратом кредита в срок?

Сгруппируем данные по столбцу `children` и применим агрегирующие функции для нахождения доли таких должников.

In [ ]:
childs_grouped = data.groupby('children')['debt'].agg(['count', 'sum'])

childs_grouped['part'] = (childs_grouped['sum'] / childs_grouped['count'] * 100).round(2)
childs_grouped = childs_grouped.sort_values(by=['part', 'children'], ascending=[False, False]).reset_index(drop=False)

childs_grouped = childs_grouped.rename(columns={'children': 'детей', 'count': 'клиентов', 'sum': 'должников', 'part': '% должников'})
childs_grouped

На первый взгляд, самыми благонадёжными являются многодетные клиенты с 5 детьми, однако, как и клиентов с 4 детьми, их количество для общей выборки незначительно (9 и 41 соотвественно). Промежуточным являются клиенты с 3 детьми, но их тоже недостаточно, чтобы сделать однозначный вывод (330 человек).

Если же обратить внимание на другие группы (0, 1, 2 ребёнка), то мы обнаружим закономерность: **с ростом числа детей растёт доля невозвратов.**

In [ ]:
general_childs_grouped = childs_grouped.drop(index=[0, 3, 5])
general_childs_grouped

Построим наглядный график для демонстрации выявленной зависимости.

In [ ]:
names = general_childs_grouped['детей']
values = general_childs_grouped['% должников']

plot_children = plt.figure(figsize=(10, 5)) 

plt.subplot(1, 1, 1)
plt.xlabel('Количество детей, шт', fontsize=10, loc='center')
plt.ylabel('Доля должников, %', fontsize=10, loc='center')
plt.title("Учеличение доли должников с ростом числа детей", fontsize=10, loc='center', color='black')

plt.plot(names, values)
plt.show()

**Промежуточный вывод**

Таким образом, мы избавились от выбросов в выборке данных и подтвердили зависимость роста доли невозвратов с ростом числа детей.

Разница между клиентами без детей и с 1, 2 ребёнком (среднее) составляет 1,8 пп.

Чаще следует одобрять:
* клиентам без детей.
  
Реже следует одобрять:
* клиентам с 1, 2 детьми.

#### 5.2. Есть ли зависимость между семейным положением и возвратом кредита в срок?

Сгруппируем данные по столбцу `family_status` и применим агрегирующие функции для нахождения доли таких должников.

In [ ]:
family_status_grouped = data.groupby('family_status')['debt'].agg(['count', 'sum'])

family_status_grouped['part'] = (family_status_grouped['sum'] / family_status_grouped['count'] * 100).round(2)
family_status_grouped = family_status_grouped.sort_values(by='part', 
                                                          ascending=False).reset_index(drop=False)
family_status_grouped

Медианной долей являются клиенты в официальном браке - 7.56%, это же самая многочисленная группа размером 12261 человек. Явных выбросов не обнаружено, т.к. в каждой группе 951 человек и больше, значит будем учитывать все группы.

Построим столбчатую диаграму, чтобы визуальной увидеть разницу в долях просрочки.

In [ ]:
names = family_status_grouped['part']
values = family_status_grouped['family_status']

plot_family_status_grouped = sns.barplot(x=names, y=values)
sns.set(rc={'figure.figsize':(10, 5)})

plot_family_status_grouped.set_xlabel('Доля должников, %')
plot_family_status_grouped.set_ylabel('Семейное положение')
plot_family_status_grouped.set_title("Распределение количества должников по семейному положению")

**Промежуточный вывод**



Таким образом, мы наблюдаем линейное снижение доли невозвратов в зависимости от категорий семейного положения.

Отталкиваясь от медианного значения доли невозвратов и самой многочисленной группы клиентов в официальном браке (12261 человек). Разница между максимальным и медианным значением составляет 2.2 пп, что является значительным показателем. Отсюда можем сделать вывод:

Чаще следует одобрять:
* клиентам в официальном браке;
* клиентам в разводе;
* клиентам в статусе вдовцов.

Реже следует одобрять:
* клиентам в гражданском браке;
* клиентам без брака.

#### 5.3. Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

Проверим, какие распределение доходов по созданным ранее категориям в столбце `total_income_category`.

In [ ]:
total_income_category_grouped = data.groupby('total_income_category')['debt'].agg(['count', 'sum'])

total_income_category_grouped['part'] = (total_income_category_grouped['sum'] / total_income_category_grouped['count'] * 100).round(2)

total_income_category_grouped.sort_values(by='part', ascending=False)

Обнаруживаем два выброса - клиенты с зарплатой 0-30000 и клиенты с зарплатой более 1000000 являются нехарактерными для выборки, т.к. их количество 22 и 25 соотвественно, при этом другое минимальное значение составляет 349 человек.

Оставим только генеральную совокупность клиентов, которых больше 349.

In [ ]:
total_income_category_grouped.drop(['E', 'A'])

**Промежуточный вывод**

Явную классификацию зависимости ухода в просрочку от уровня дохода обозначить трудно, т.к. категории включают в себя определённые подмножества диапозонов доходов.

Чаще следует одобрять:
* клиентам с уровнем доходов категории D (30001-50000);
* клиентам с уровнем доходов категории B (200001-1000000).

Реже следует одобрять:
* клиентам с уровнем доходов категории С (50001-200000).

#### 5.4. Как разные цели кредита влияют на его возврат в срок?

Цели кредитов бывают самыми разными. Выполним группировку по цели, посмотрим на количество клиентов, которые берут кредит с этой целью, количество должников и долю по отношению к этой же группе.

In [ ]:
purposes_grouped = data.groupby('purpose_category')['debt'].agg(['count', 'sum'])

purposes_grouped['part'] = (purposes_grouped['sum'] / purposes_grouped['count'] * 100).round(2)
purposes_grouped = purposes_grouped.sort_values(by='part')
display(purposes_grouped)

purposes_grouped.median()

Мы можем обнаружить, что медианой количества для каждой цели является 603 человека, из которых в просрочку уходят 45.5 человек или 8.08%.

Самая низкая просрочка с целью - операции с недвижимостью. 

Самая высокая просрочка с целью - операции с автомобилем.

**Промежуточный вывод**

Таким образом, все 4 категории имеют риск не вернуть заёмные средства, однако отталкиваясь от медины следует чуть чаще одобрять клиентам с целями:
* операции с недвижимостью;
* проведение свадьбы.

И реже клиентам с целями:

* получение образования;
* операции с автомобилем.

#### 5.5. Приведите возможные причины появления пропусков в исходных данных.

Пропуски в колонках `income_type` и `days_employed` могли возникнуть по объективной причине, ведь среди клиентов есть безработные, которые не имеют постоянного дохода и трудовой стаж.

В целом пропуски могут быть по причине технический сбоев, человеческого фактора, отсутствия обязательного пункта в анкете и по ряду других причин.

#### 5.6. Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

Заполнение пропусков зависит от задач исследования, ведь мы можем, наоборот, искать количество этих пропусков.

Количественные пропуски заполняют либо расчётными, либо медианными значениями. Медианное значение позволяет данным быть утойчивыми к выбросам в отличие от среднего арифмитического и показывать более характерные значения для выборки. В нашем случае значение ячеек столбца не зависили от собственного столбца, но имели зависимость от другого (тип занятости) - это позволило ещё более тонко подойти к заполнению пропусков и не исказить датафрейм.

[_назад к содержанию_](#content)

<a class='anchor' id='result'></a>
### 6. Результаты исследования

В ходе исследования были выдвинуты несколько гипотез с целью улучшения модели скоринговой системы:

1. Есть ли зависимость между количеством детей и возвратом кредита в срок?
2. Есть ли зависимость между семейным положением и возвратом кредита в срок?
3. Есть ли зависимость между уровнем дохода и возвратом кредита в срок?
4. Как разные цели кредита влияют на его возврат в срок?
   
Была выполнена предобработка данных, построены таблицы и графики, рассчитывающие и отображающие долю ухода клиентов в просрочку по определённному признаку.

Кроме того, по доп. заданию построен интерактивный дашборд.

**Результаты по задачам**

<div class="alert alert-info">
<b> 1. Есть ли зависимость между количеством детей и возвратом кредита в срок? </div>

Да, с ростом числа детей растёт уровень невозврата:
* нет детей: 7.54%
* один ребёнок: 9.23
* два ребёнка: 9.45%

<div class="alert alert-info">
<b> 2. Есть ли зависимость между семейным положением и возвратом кредита в срок? </div>
    
Да, отталкиваясь от медианного значения доли невозвратов и самой многочисленной группы клиентов в официальном браке (12261 человек) меньше всего должников в группах:
   * официальный брак
   * в разводе
   * статус вдовцов
    
Реже следует одобрять:
* клиентам в гражданском браке;
* клиентам без брака.

<div class="alert alert-info">
<b> 3. Есть ли зависимость между уровнем дохода и возвратом кредита в срок? </div>
    
Явную зависимость обнаружить сложно, т.к. уникальных значений уровня доходов большое множество, однако можно выделить следующее.
    
Чаще следует одобрять:
* клиентам с уровнем доходов 30001-50000;
* клиентам с уровнем доходов 200001-1000000.

Реже следует одобрять:
* клиентам с уровнем доходов 50001-200000.

<div class="alert alert-info">
<b> 4. Как разные цели кредита влияют на его возврат в срок? </div>
    
Все 4 категории по целям взятия кредита имеют риск не вернуть заёмные средства, при этом можно сказать, что чаще следует одобрять клиентам с целями:
* операции с недвижимостью;
* проведение свадьбы.

И реже клиентам с целями:
* получение образования;
* операции с автомобилем.

[_назад к содержанию_](#content)

<a class='anchor' id='scoring'></a>
### 7. Рекомендации по улучшению скоринга 

Обобщив результаты 4 гипотез, можно составить **портрет идеального заёмщика** по совокупоности признаков:
* нет детей;
* статусы брака: официальный, вдовдствующий или в разводе;
* уровень доходов 30001-50000 или 200001-1000000;
* кредит требуется для операций с недвижимостью или для проведения свадьбы.

При одновременном выволнении вышеназванных критериев шанс возврата становится выше, чем в среднем по всем клиентам.

По такому же принцпипу можно составить **потрет высокорискового заёмщика**:
* больше 1 ребёнка;
* не в браке или в гражданском браке;
* уровень доходов от 50001 до 200000;
* кредится требуется для получения образования или операций с автомобилем.

[_назад к содержанию_](#content)

<a class='anchor' id='recomendation'></a>
### 8. Рекомендации по сбору данных

Сделать обязательными поля при заполнении анкеты:
* days_employed
* total_income

Определить наличие аномали в столбцах:
* total_income (более 2 миллионов)
* children (значения -1 и 20)

Это позволит провести более качественное исследование.

[_назад к содержанию_](#content)